In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/screen_time_mental_health.csv")
df

,subject_id,sex,screen_time_index,est_leisure_screen_hours,sleep_quality_index,avg_sleep_hours,midsleep_weekend_hours,social_jetlag_hours,bdi_total,depressed
0,1,Boy,3.333,4.64,2.25,8.36,4.58,1.92,6,0
1,2,Girl,3.333,4.07,2.50,7.31,5.42,2.25,7,0
2,3,Boy,3.333,4.07,2.00,8.75,5.50,2.62,27,1
3,4,Boy,4.333,6.07,1.50,8.17,6.62,3.96,22,1
4,5,Boy,1.667,0.50,2.50,8.88,4.12,1.79,3,0
...,...,...,...,...,...,...,...,...,...,...
4805,4806,Girl,2.333,2.07,1.50,8.76,4.08,2.00,20,1
4806,4807,Boy,4.333,6.07,1.00,6.60,7.46,3.46,6,0
4807,4808,Girl,2.333,2.07,1.25,6.61,5.25,1.62,1,0
4808,4809,Girl,3.667,4.64,1.25,7.15,5.54,2.83,0,0


## Paso 3: Almacenar la información

In [4]:
import sqlite3

nombre_base_datos = 'screen_time_mental_health.db'
conn = sqlite3.connect(nombre_base_datos)

df.to_sql('screen_time_mental_health', conn, if_exists='replace', index=False)

conn.close()

Haremos algunas consultas:
- La primera consulta será para obtener la cantidad de registros en la tabla 'screen_time_mental_health'.

In [5]:
conn = sqlite3.connect(nombre_base_datos)
cursor = conn.cursor()

query_1 = "SELECT COUNT(*) FROM screen_time_mental_health"

cursor.execute(query_1)

total_registros = cursor.fetchone()[0]

print(f"Cantidad total de registros: {total_registros}")

Cantidad total de registros: 4810


- La segunda consulta será para ver la cantidad de chicos ('Boy) y chicas ('Girl) que tenemos en la base de datos.


In [6]:
query_2 = """
SELECT sex, COUNT(*) 
FROM screen_time_mental_health 
GROUP BY sex;
"""

cursor.execute(query_2)
resultados = cursor.fetchall() 

for sex, cantidad in resultados:
    print(f"Sexo {sex}: {cantidad}")


Sexo Boy: 2446
Sexo Girl: 2364


- La tercera consulta será el valor promedio, máximo y mínimo de la columna 'bdi_total'


In [32]:
query_3 = """

SELECT

AVG(bdi_total) AS promedio,

MAX(bdi_total) AS maximo,

MIN(bdi_total) AS minimo

FROM screen_time_mental_health;

"""



cursor.execute(query_3)

promedio, maximo, minimo = cursor.fetchone()



print(f"Estadísticas Promedio: {promedio:.2f}, Máximo: {maximo}, Mínimo: {minimo}")


Estadísticas Promedio: 7.27, Máximo: 51, Mínimo: 0


- La cuarta consulta es el valor promedio, máximo y mínimo de la columna 'bdi_total', pero a diferencia de la tercera consulta, esta vez estará agrupada por sexo.

In [31]:
query_4 = """
SELECT
    sex,
    AVG(bdi_total) AS promedio,
    MAX(bdi_total) AS maximo,
    MIN(bdi_total) AS minimo
FROM screen_time_mental_health
GROUP BY sex;
"""

cursor.execute(query_4)
resultados = cursor.fetchall()

for sexo, promedio, maximo, minimo in resultados:
    print(f"Sexo {sexo}: Promedio: {promedio:.2f}, Máximo: {maximo}, Mínimo: {minimo}")


Sexo Boy: Promedio: 5.22, Máximo: 43, Mínimo: 0
Sexo Girl: Promedio: 9.40, Máximo: 51, Mínimo: 0


- La quinta y última consulta es sobre la clasificación de los usuarios por su Tiempo promedio en pantalla y el promedio de 'bd_total' de cada clasificación.

In [29]:
query_5 = """
SELECT 
    CASE 
        WHEN screen_time_index == 1 THEN 'Tiempo promedio en pantalla < 1h'
        WHEN screen_time_index == 2 THEN 'Tiempo promedio en pantalla 1-2h'
        WHEN screen_time_index == 3 THEN 'Tiempo promedio en pantalla 3-4h'
        WHEN screen_time_index == 4 THEN 'Tiempo promedio en pantalla 5-6h'
        WHEN screen_time_index == 5 THEN 'Tiempo promedio en pantalla 7-8h'
        ELSE 'Tiempo promedio en pantalla > 8h'
    END AS categoria_pantalla,
    COUNT(*) AS cantidad_usuarios,
    AVG(bdi_total) AS promedio_bdi
FROM screen_time_mental_health
GROUP BY categoria_pantalla
ORDER BY 
    CASE 
        WHEN categoria_pantalla == 'Tiempo promedio en pantalla < 1h' THEN 0 
        ELSE categoria_pantalla
    END ASC;
"""

cursor.execute(query_5)
for categoria, cantidad, promedio_bdi in cursor.fetchall():
    print(f"Categoría: {categoria} - Usuarios: {cantidad} - BDI Promedio: {promedio_bdi:.2f}")

Categoría: Tiempo promedio en pantalla < 1h - Usuarios: 17 - BDI Promedio: 7.47
Categoría: Tiempo promedio en pantalla 1-2h - Usuarios: 452 - BDI Promedio: 5.68
Categoría: Tiempo promedio en pantalla 3-4h - Usuarios: 501 - BDI Promedio: 7.25
Categoría: Tiempo promedio en pantalla 5-6h - Usuarios: 319 - BDI Promedio: 7.76
Categoría: Tiempo promedio en pantalla 7-8h - Usuarios: 171 - BDI Promedio: 8.49
Categoría: Tiempo promedio en pantalla > 8h - Usuarios: 3350 - BDI Promedio: 7.38
